# Workshop Databricks — ML intermediário
## Risco de crédito + MLOps (guia único)

| Módulo | Resultado |
|---|---|
| Configuração | Catálogo, schema, prefixo e caminho dos dados |
| 1 — Dados | Tabelas Delta no Unity Catalog |
| 2 — Feature Engineering | Feature table em UC (`FeatureEngineeringClient`) |
| 3 — Treinamento | Comparação de modelos + MLflow + Challenger |
| 4 — Promoção MLOps | Gate de métrica e alias Champion |
| 5 — Model Serving | Endpoint + invocação em tempo real |
| 6 — Monitoramento | Inference tables e próximos passos |
| Opcional — Model-inside-model | Roteamento multi-modelo em um PyFunc |

**Duração sugerida:** 1h30 (pré-execute dados/features/serving se for ao vivo).


## Modelo de dados

Caso de uso: **decisão de concessão / risco de crédito**.

```text
customer_gold (cust_id)
   ├── telco_gold (cust_id)
   ├── fund_trans_gold (cust_id)
   └── credit_bureau_gold (cust_id)  → label defaulted
            │
            ▼
   credit_decisioning_features (cust_id)  → treino / serving
```

| Tabela | Papel | Principais campos |
|---|---|---|
| `customer_gold` | Perfil do cliente | `cust_id`, tenure, renda, saldo |
| `telco_gold` | Sinais alternativos | atrasos de fatura, prepaid |
| `fund_trans_gold` | Comportamento transacional | envios/recebimentos 3/6/12m |
| `credit_bureau_gold` | Histórico de crédito / label | `CREDIT_DAY_OVERDUE` |
| `credit_decisioning_features` | Feature table (UC) | features consolidadas por `cust_id` |


---
## Configuração inicial

Preencha **apenas as variáveis** abaixo (sem widgets).

- `CATALOGO` / `SCHEMA`: destino Unity Catalog
- `PREFIXO`: evita colisão entre participantes (ex.: `carol_`)
- `CAMINHO_DADOS`: pasta com os CSVs (Volume UC recomendado)

Exemplo de Volume:

```text
/Volumes/workspace_carolina_ferreira_catalog/ml_workshop/dados
```


In [0]:
# ============================================================
# Preencha apenas estas variáveis
# ============================================================
CATALOGO = "workspace_carolina_ferreira_catalog"
SCHEMA = "ml_workshop"
PREFIXO = "ws_"
CAMINHO_DADOS = "/Volumes/workspace_carolina_ferreira_catalog/ml_workshop/dados"

# Controles de execução (workshop ao vivo / teste)
RUN_SERVING = True
RUN_OPTIONAL = False

assert CATALOGO and SCHEMA and PREFIXO and CAMINHO_DADOS, "Preencha CATALOGO, SCHEMA, PREFIXO e CAMINHO_DADOS"

identificador_schema = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {identificador_schema}")

print(f"Destino: {identificador_schema} | Prefixo: {PREFIXO}")
print(f"Dados: {CAMINHO_DADOS}")
print(f"RUN_SERVING={RUN_SERVING} | RUN_OPTIONAL={RUN_OPTIONAL}")

---
## Módulo 1 — Criar tabelas Delta

Carrega apenas as tabelas **raw** e grava Delta no schema do workshop.
A feature table será criada no Módulo 2 com `FeatureEngineeringClient`.


In [0]:
from pathlib import Path
import re

RAW_TABLES = {
    "customer_gold",
    "telco_gold",
    "fund_trans_gold",
    "credit_bureau_gold",
}

dados_dir = Path(CAMINHO_DADOS)
if not dados_dir.exists():
    raise FileNotFoundError(f"Pasta de dados não encontrada: {dados_dir}")

csv_files = sorted(dados_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Nenhum CSV em: {dados_dir}")


def normalizar_nome(nome: str) -> str:
    nome = re.sub(r"[^a-zA-Z0-9_]", "_", nome.strip().lower())
    return re.sub(r"_+", "_", nome).strip("_")


resultados = []
for arquivo in csv_files:
    stem = normalizar_nome(arquivo.stem)
    if stem not in RAW_TABLES:
        print(f"Ignorando {arquivo.name} (não é tabela raw)")
        continue
    nome_tabela = normalizar_nome(f"{PREFIXO}{stem}")
    tabela_destino = f"{identificador_schema}.{nome_tabela}"
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(str(arquivo))
    )
    for c in list(df.columns):
        if c.lower() == "cust_id" and c != "cust_id":
            df = df.withColumnRenamed(c, "cust_id")
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(tabela_destino)
    )
    resultados.append(
        {
            "arquivo_csv": arquivo.name,
            "tabela_destino": tabela_destino,
            "linhas": spark.table(tabela_destino).count(),
            "colunas": len(df.columns),
        }
    )

display(spark.createDataFrame(resultados))

---
## Módulo 2 — Feature Engineering in Unity Catalog

Boas práticas atuais:

1. Qualquer tabela Delta com **primary key** pode ser feature table no UC
2. Use `databricks-feature-engineering` + `FeatureEngineeringClient` (Workspace Feature Store está deprecado)
3. No treino, use `FeatureLookup` + `create_training_set` e `fe.log_model` para linhagem


In [0]:
# Instalação (interativo). Em Jobs serverless, as deps podem vir no environment.
import importlib
import subprocess
import sys

required = [
    ("databricks.feature_engineering", "databricks-feature-engineering"),
    ("mlflow", "mlflow>=3.0"),
    ("sklearn", "scikit-learn"),
    ("databricks.sdk", "databricks-sdk"),
]
missing = []
for mod, pkg in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Instalado:", missing)
else:
    print("Dependências já disponíveis")

In [0]:
# Reaplicar config (após installs / em novas sessões)
CATALOGO = "workspace_carolina_ferreira_catalog"
SCHEMA = "ml_workshop"
PREFIXO = "ws_"
CAMINHO_DADOS = "/Volumes/workspace_carolina_ferreira_catalog/ml_workshop/dados"
RUN_SERVING = True
RUN_OPTIONAL = False
identificador_schema = f"{CATALOGO}.{SCHEMA}"
print(identificador_schema, PREFIXO, RUN_SERVING, RUN_OPTIONAL)

In [0]:
from datetime import date
from pyspark.sql.functions import col
from pyspark.sql import functions as F
import plotly.express as px

customer_gold_features = (
    spark.table(f"{identificador_schema}.{PREFIXO}customer_gold")
    .withColumn("age", int(date.today().year) - col("birth_year"))
    .select(
        "cust_id", "education", "marital_status", "months_current_address",
        "months_employment", "is_resident", "tenure_months", "product_cnt",
        "tot_rel_bal", "revenue_tot", "revenue_12m", "income_annual", "tot_assets",
        "overdraft_balance_amount", "overdraft_number", "total_deposits_number",
        "total_deposits_amount", "total_equity_amount", "total_UT", "customer_revenue",
        "age", "avg_balance", "num_accs", "balance_usd", "available_balance_usd",
    )
    .dropDuplicates(["cust_id"])
)

telco_gold_features = (
    spark.table(f"{identificador_schema}.{PREFIXO}telco_gold")
    .select(
        "cust_id", "is_pre_paid", "number_payment_delays_last12mo",
        "pct_increase_annual_number_of_delays_last_3_year", "phone_bill_amt",
        "avg_phone_bill_amt_lst12mo",
    )
    .dropDuplicates(["cust_id"])
)

fund_trans_gold_features = spark.table(f"{identificador_schema}.{PREFIXO}fund_trans_gold").dropDuplicates(["cust_id"])
for window in ["12m", "6m", "3m"]:
    fund_trans_gold_features = (
        fund_trans_gold_features
        .withColumn(f"tot_txn_cnt_{window}", col(f"sent_txn_cnt_{window}") + col(f"rcvd_txn_cnt_{window}"))
        .withColumn(f"tot_txn_amt_{window}", col(f"sent_txn_amt_{window}") + col(f"rcvd_txn_amt_{window}"))
    )

fund_trans_gold_features = (
    fund_trans_gold_features
    .withColumn(
        "ratio_txn_amt_3m_12m",
        F.when(col("tot_txn_amt_12m") == 0, 0).otherwise(col("tot_txn_amt_3m") / col("tot_txn_amt_12m")),
    )
    .withColumn(
        "ratio_txn_amt_6m_12m",
        F.when(col("tot_txn_amt_12m") == 0, 0).otherwise(col("tot_txn_amt_6m") / col("tot_txn_amt_12m")),
    )
    .na.fill(0)
)

feature_df = (
    customer_gold_features
    .join(telco_gold_features, "cust_id", "left")
    .join(fund_trans_gold_features, "cust_id", "left")
)
display(feature_df.limit(20))
print(f"Features: {feature_df.count()} linhas | {len(feature_df.columns)} colunas")

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()
feature_table_name = f"{identificador_schema}.{PREFIXO}credit_decisioning_features"

try:
    fe.create_table(
        name=feature_table_name,
        primary_keys=["cust_id"],
        df=feature_df,
        description="Features de credit decisioning (workshop ML intermediário).",
    )
except Exception as e:
    msg = str(e).lower()
    if "already exists" in msg or "table_or_view_already_exists" in msg:
        fe.write_table(name=feature_table_name, df=feature_df, mode="overwrite")
    else:
        raise

print(f"Feature table pronta: {feature_table_name}")
display(fe.read_table(name=feature_table_name).limit(10))

---
## Módulo 3 — Treinamento + MLflow + Challenger

1. Monta label `defaulted` (atraso > 60 dias no bureau)
2. Cria training set via `FeatureLookup`
3. Balanceia classes (over/under sampling didático)
4. Treina 5 modelos sklearn e loga no MLflow
5. Registra o melhor no Unity Catalog com alias **Challenger**


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import plotly.express as px

fe = FeatureEngineeringClient()
feature_table_name = f"{identificador_schema}.{PREFIXO}credit_decisioning_features"

raw_bureau = spark.table(f"{identificador_schema}.{PREFIXO}credit_bureau_gold")
cust_col = "CUST_ID" if "CUST_ID" in raw_bureau.columns else "cust_id"
credit_bureau_label = (
    raw_bureau
    .withColumn("cust_id", col(cust_col))
    .withColumn("defaulted", F.when(col("CREDIT_DAY_OVERDUE") > 60, 1).otherwise(0))
    .select("cust_id", "defaulted")
)

px.pie(
    credit_bureau_label.groupBy("defaulted").count().toPandas(),
    values="count",
    names="defaulted",
    title="Proporção de inadimplência (antes do balanceamento)",
).show()

training_set = fe.create_training_set(
    df=credit_bureau_label,
    feature_lookups=[FeatureLookup(table_name=feature_table_name, lookup_key="cust_id")],
    label="defaulted",
)
training_dataset = training_set.load_df().na.fill(0)
print(f"Training set: {training_dataset.count()} linhas")

In [0]:
from pyspark.sql.functions import col
import plotly.express as px

major_df = training_dataset.filter(col("defaulted") == 0)
minor_df = training_dataset.filter(col("defaulted") == 1)
oversampled_df = minor_df.union(minor_df)
undersampled_df = major_df.sample(oversampled_df.count() / major_df.count() * 3, 42)
train_df = undersampled_df.unionAll(oversampled_df).drop("cust_id").na.fill(0)

tabela_treino = f"{identificador_schema}.{PREFIXO}credit_risk_train_df"
train_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(tabela_treino)
train_df = spark.table(tabela_treino)

px.pie(
    train_df.groupBy("defaulted").count().toPandas(),
    values="count",
    names="defaulted",
    title="Proporção de inadimplência (após balanceamento)",
).show()
print(f"Tabela de treino: {tabela_treino}")

In [0]:
import mlflow
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
import pandas as pd

mlflow.set_registry_uri("databricks-uc")
experiment_path = f"/Users/carolina.ferreira@databricks.com/{PREFIXO}ml_workshop_treinamento"
mlflow.set_experiment(experiment_path)

credit_risk_pdf = spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df").toPandas()
X_cr = credit_risk_pdf.drop(columns=["defaulted"])
y_cr = credit_risk_pdf["defaulted"]
X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(
    X_cr, y_cr, test_size=0.2, random_state=42
)

models_cr = {
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "Logistic Regression": LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0, max_iter=1000, random_state=42
    ),
}

results_cr = []
for name, m in models_cr.items():
    with mlflow.start_run(run_name=f"{name}_credit_risk") as run:
        m.fit(X_train_cr, y_train_cr)
        y_pred = m.predict(X_test_cr)
        y_prob = m.predict_proba(X_test_cr)[:, 1]
        acc = accuracy_score(y_test_cr, y_pred)
        roc = roc_auc_score(y_test_cr, y_prob)
        f1 = f1_score(y_test_cr, y_pred)
        mlflow.log_params({k: str(v) for k, v in m.get_params().items()})
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", roc)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(m, artifact_path="model", input_example=X_test_cr.iloc[:5])
        results_cr.append({
            "Model": name, "Accuracy": round(acc, 4), "ROC AUC": round(roc, 4),
            "F1": round(f1, 4), "run_id": run.info.run_id, "model_obj": m,
        })
        print(f"{name}: Accuracy={acc:.4f}, ROC AUC={roc:.4f}, F1={f1:.4f}")

comparison_cr_df = pd.DataFrame([{k: v for k, v in r.items() if k != "model_obj"} for r in results_cr])
comparison_cr_df = comparison_cr_df.sort_values("ROC AUC", ascending=False)
display(comparison_cr_df)

best = max(results_cr, key=lambda r: r["ROC AUC"])
best_model_name = best["Model"]
best_run_id = best["run_id"]
best_model = best["model_obj"]
best_auc = float(best["ROC AUC"])
print(f"Melhor modelo: {best_model_name} (ROC AUC={best_auc})")

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import mlflow
from mlflow import MlflowClient

fe = FeatureEngineeringClient()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
model_name = f"{identificador_schema}.{PREFIXO}credit_classification"

raw_bureau = spark.table(f"{identificador_schema}.{PREFIXO}credit_bureau_gold")
cust_col = "CUST_ID" if "CUST_ID" in raw_bureau.columns else "cust_id"
credit_bureau_label = (
    raw_bureau
    .withColumn("cust_id", col(cust_col))
    .withColumn("defaulted", F.when(col("CREDIT_DAY_OVERDUE") > 60, 1).otherwise(0))
    .select("cust_id", "defaulted")
)
feature_table_name = f"{identificador_schema}.{PREFIXO}credit_decisioning_features"
training_set = fe.create_training_set(
    df=credit_bureau_label,
    feature_lookups=[FeatureLookup(table_name=feature_table_name, lookup_key="cust_id")],
    label="defaulted",
)

with mlflow.start_run(run_name=f"register_{best_model_name}") as run:
    mlflow.log_metric("roc_auc", best_auc)
    model_info = fe.log_model(
        model=best_model,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=model_name,
        input_example=X_test_cr.iloc[:5],
    )

versions = client.search_model_versions(f"name='{model_name}'")
latest_version = str(max(int(v.version) for v in versions))
client.set_registered_model_alias(model_name, "Challenger", latest_version)
challenger_auc = best_auc
challenger_version = latest_version
print(f"Registrado: {model_name} v{challenger_version} @Challenger | ROC AUC={challenger_auc}")
print(f"model_uri: {model_info.model_uri}")

---
## Módulo 4 — Promoção MLOps (Challenger → Champion)

Gate simples e didático:

- Se **não existe Champion**, promove o Challenger
- Se existe, promove só se o Challenger tiver **ROC AUC ≥ Champion** e acima de um mínimo


In [0]:
from mlflow import MlflowClient
import mlflow

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
model_name = f"{identificador_schema}.{PREFIXO}credit_classification"
MIN_AUC = 0.60

challenger = client.get_model_version_by_alias(model_name, "Challenger")
challenger_version = challenger.version
challenger_run = client.get_run(challenger.run_id)
run_auc = challenger_run.data.metrics.get("roc_auc")
challenger_auc_gate = float(run_auc) if run_auc is not None else float(globals().get("challenger_auc", 0))

promote = False
reason = ""
try:
    champion = client.get_model_version_by_alias(model_name, "Champion")
    champion_run = client.get_run(champion.run_id)
    champion_auc = float(champion_run.data.metrics.get("roc_auc", 0))
    if challenger_auc_gate >= champion_auc and challenger_auc_gate >= MIN_AUC:
        promote = True
        reason = f"Challenger AUC {challenger_auc_gate:.4f} >= Champion AUC {champion_auc:.4f} e >= {MIN_AUC}"
    else:
        reason = f"Não promoveu: Challenger {challenger_auc_gate:.4f} vs Champion {champion_auc:.4f} (mín {MIN_AUC})"
except Exception:
    if challenger_auc_gate >= MIN_AUC:
        promote = True
        reason = f"Sem Champion prévio; Challenger AUC {challenger_auc_gate:.4f} >= {MIN_AUC}"
    else:
        reason = f"Sem Champion e AUC {challenger_auc_gate:.4f} < mínimo {MIN_AUC}"

if promote:
    client.set_registered_model_alias(model_name, "Champion", challenger_version)
    print(f"PROMOVIDO → Champion v{challenger_version}")
else:
    print("Mantido Challenger sem promoção")
print(reason)

champ = client.get_model_version_by_alias(model_name, "Champion")
print(f"Champion atual: v{champ.version}")

In [0]:
import mlflow
import pandas as pd

model_name = f"{identificador_schema}.{PREFIXO}credit_classification"
loaded_model = mlflow.sklearn.load_model(f"models:/{model_name}@Champion")
predictions = loaded_model.predict(X_test_cr)
probabilities = loaded_model.predict_proba(X_test_cr)[:, 1]
results_batch = pd.DataFrame({
    "actual": y_test_cr.values,
    "predicted": predictions,
    "probability_default": probabilities,
})
print(f"Batch inference: {len(results_batch)} registros")
display(spark.createDataFrame(results_batch.head(20)))

---
## Módulo 5 — Model Serving

Cria (ou atualiza) um endpoint apontando para a versão **Champion**.

> Em workshops ao vivo, pré-crie o endpoint: o provisionamento pode levar vários minutos.


In [0]:
if not RUN_SERVING:
    print('RUN_SERVING=False — pulando serving')
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
    from mlflow import MlflowClient
    import mlflow

    mlflow.set_registry_uri("databricks-uc")
    client_mlflow = MlflowClient()
    model_name = f"{identificador_schema}.{PREFIXO}credit_classification"
    endpoint_name = f"{PREFIXO}credit-risk-serving".replace("_", "-")
    model_version = client_mlflow.get_model_version_by_alias(model_name, "Champion").version
    print(f"Modelo: {model_name} | Champion v{model_version}")
    print(f"Endpoint: {endpoint_name}")

    w = WorkspaceClient()
    entity = ServedEntityInput(
        name=f"{PREFIXO}credit-entity".replace("_", "-"),
        entity_name=model_name,
        entity_version=model_version,
        workload_size="Small",
        scale_to_zero_enabled=True,
    )

    try:
        w.serving_endpoints.create_and_wait(
            name=endpoint_name,
            config=EndpointCoreConfigInput(served_entities=[entity]),
        )
        print(f"Endpoint criado: {endpoint_name}")
    except Exception as e:
        msg = str(e).lower()
        if "already exists" in msg or "resource_already_exists" in msg:
            w.serving_endpoints.update_config_and_wait(name=endpoint_name, served_entities=[entity])
            print(f"Endpoint atualizado: {endpoint_name}")
        else:
            raise

    info = w.serving_endpoints.get(name=endpoint_name)
    print(f"Estado: {info.state.ready}")

In [0]:
if not RUN_SERVING:
    print('RUN_SERVING=False — pulando serving')
else:
    import mlflow.deployments
    import pandas as pd

    df_exemplo = (
        spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df")
        .drop("defaulted")
        .limit(5)
        .toPandas()
    )
    print("Payload:")
    display(spark.createDataFrame(df_exemplo))

    deploy_client = mlflow.deployments.get_deploy_client("databricks")
    response = deploy_client.predict(
        endpoint=endpoint_name,
        inputs={"dataframe_split": df_exemplo.to_dict(orient="split")},
    )
    print("Resposta:", response)

    preds = response["predictions"] if isinstance(response, dict) and "predictions" in response else response
    resultado = pd.concat([
        df_exemplo.reset_index(drop=True),
        pd.DataFrame({"predicao_risco": preds}),
    ], axis=1)
    display(spark.createDataFrame(resultado))

---
## Módulo 6 — Monitoramento e próximos passos MLOps

Para fechar o ciclo intermediário:

1. **Inference tables** no endpoint (requests/responses para auditoria e drift)
2. **Data Profiling / Lakehouse Monitoring** na feature table e nas predições
3. **Lakeflow Jobs** para retreino periódico + re-promote Champion
4. **DABs** para versionar job + serving como código

No UI: **Serving → endpoint → Inference tables → Enable**. Depois compare a distribuição de `probability_default` treino vs produção.


In [0]:
from mlflow import MlflowClient
import mlflow

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
model_name = f"{identificador_schema}.{PREFIXO}credit_classification"

checklist = {
    "feature_table": f"{identificador_schema}.{PREFIXO}credit_decisioning_features",
    "train_table": f"{identificador_schema}.{PREFIXO}credit_risk_train_df",
    "model": model_name,
    "champion": client.get_model_version_by_alias(model_name, "Champion").version,
    "endpoint": endpoint_name,
}
for k, v in checklist.items():
    print(f"✓ {k}: {v}")

---
# Opcional — Model-inside-model (roteamento)

Empacota **dois classificadores** em um único PyFunc e roteia por `modelo_alvo`:

- `conservador` → Logistic Regression
- `agressivo` → Gradient Boosting
- `auto` → regra simples nas features

Execute só se sobrar tempo após o fluxo principal.


In [0]:
if not RUN_OPTIONAL:
    print('RUN_OPTIONAL=False — pulando opcional')
else:
    import mlflow
    import pandas as pd
    from mlflow.models import infer_signature
    from mlflow import MlflowClient
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    import tempfile, os, pickle

    mlflow.set_registry_uri("databricks-uc")
    mlflow.set_experiment(f"/Users/carolina.ferreira@databricks.com/{PREFIXO}ml_workshop_router")

    df_treino = spark.table(f"{identificador_schema}.{PREFIXO}credit_risk_train_df").toPandas()
    X = df_treino.drop(columns=["defaulted"])
    y = df_treino["defaulted"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo_conservador = LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, C=1.0, max_iter=1000, random_state=42
    )
    modelo_conservador.fit(X_train, y_train)
    auc_cons = roc_auc_score(y_test, modelo_conservador.predict_proba(X_test)[:, 1])

    modelo_agressivo = GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    )
    modelo_agressivo.fit(X_train, y_train)
    auc_agr = roc_auc_score(y_test, modelo_agressivo.predict_proba(X_test)[:, 1])
    print(f"Conservador AUC={auc_cons:.4f} | Agressivo AUC={auc_agr:.4f}")

    tmp_dir = tempfile.mkdtemp()
    conservador_path = os.path.join(tmp_dir, "modelo_conservador.pkl")
    agressivo_path = os.path.join(tmp_dir, "modelo_agressivo.pkl")
    colunas_path = os.path.join(tmp_dir, "columns.txt")
    with open(conservador_path, "wb") as f:
        pickle.dump(modelo_conservador, f)
    with open(agressivo_path, "wb") as f:
        pickle.dump(modelo_agressivo, f)
    with open(colunas_path, "w") as f:
        f.write(",".join(X.columns.tolist()))


    class CreditRiskRouter(mlflow.pyfunc.PythonModel):
        def load_context(self, context):
            import pickle
            with open(context.artifacts["modelo_conservador"], "rb") as f:
                self.modelo_conservador = pickle.load(f)
            with open(context.artifacts["modelo_agressivo"], "rb") as f:
                self.modelo_agressivo = pickle.load(f)
            with open(context.artifacts["columns_file"], "r") as f:
                self.expected_columns = f.read().strip().split(",")

        def _preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
            for col_name in self.expected_columns:
                if col_name not in df.columns:
                    df[col_name] = 0.0
            df = df[self.expected_columns].copy().fillna(0)
            for c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).clip(-1e6, 1e6)
            return df

        def _decide_route(self, row_features: pd.Series) -> str:
            return "agressivo" if row_features.mean() > 0.3 else "conservador"

        def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
            if "modelo_alvo" in model_input.columns:
                routes = model_input["modelo_alvo"].fillna("auto").values
                model_input = model_input.drop(columns=["modelo_alvo"])
            else:
                routes = ["auto"] * len(model_input)
            processed = self._preprocess(model_input)
            predictions, models_used = [], []
            for i in range(len(processed)):
                route = routes[i]
                if route not in ("conservador", "agressivo"):
                    route = self._decide_route(processed.iloc[i])
                row = processed.iloc[[i]]
                model = self.modelo_conservador if route == "conservador" else self.modelo_agressivo
                predictions.append(float(model.predict_proba(row)[0, 1]))
                models_used.append(route)
            return pd.DataFrame({"risk_score": predictions, "modelo_utilizado": models_used})


    router_model_name = f"{identificador_schema}.{PREFIXO}credit_risk_router"
    df_input_example = X_test.head(3).copy()
    df_input_example["modelo_alvo"] = ["conservador", "agressivo", "auto"]
    df_output_example = pd.DataFrame({
        "risk_score": [0.12, 0.78, 0.45],
        "modelo_utilizado": ["conservador", "agressivo", "agressivo"],
    })
    signature = infer_signature(df_input_example, df_output_example)

    with mlflow.start_run(run_name="credit_risk_router_classifier"):
        model_info = mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=CreditRiskRouter(),
            artifacts={
                "modelo_conservador": conservador_path,
                "modelo_agressivo": agressivo_path,
                "columns_file": colunas_path,
            },
            signature=signature,
            input_example=df_input_example,
            pip_requirements=["scikit-learn", "mlflow", "pandas", "numpy"],
            registered_model_name=router_model_name,
        )

    client_mlflow = MlflowClient()
    versions = client_mlflow.search_model_versions(f"name='{router_model_name}'")
    latest_version = str(max(int(v.version) for v in versions))
    client_mlflow.set_registered_model_alias(router_model_name, "Champion", latest_version)
    print(f"Router: {router_model_name} v{latest_version} @Champion")

In [0]:
if not RUN_OPTIONAL:
    print('RUN_OPTIONAL=False — pulando opcional')
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
    import mlflow.deployments
    import pandas as pd

    router_endpoint_name = f"{PREFIXO}credit-risk-router".replace("_", "-")
    w = WorkspaceClient()
    entity = ServedEntityInput(
        name=f"{PREFIXO}credit-router-entity".replace("_", "-"),
        entity_name=router_model_name,
        entity_version=latest_version,
        workload_size="Small",
        scale_to_zero_enabled=True,
    )
    try:
        w.serving_endpoints.create_and_wait(
            name=router_endpoint_name,
            config=EndpointCoreConfigInput(served_entities=[entity]),
        )
    except Exception as e:
        if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
            w.serving_endpoints.update_config_and_wait(name=router_endpoint_name, served_entities=[entity])
        else:
            raise

    df_teste_router = X_test.head(6).copy()
    df_teste_router["modelo_alvo"] = ["conservador", "conservador", "agressivo", "agressivo", "auto", "auto"]
    deploy_client = mlflow.deployments.get_deploy_client("databricks")
    response = deploy_client.predict(
        endpoint=router_endpoint_name,
        inputs={"dataframe_split": df_teste_router.to_dict(orient="split")},
    )
    resultado_df = pd.DataFrame(response["predictions"] if "predictions" in response else response)
    display(pd.concat([df_teste_router[["modelo_alvo"]].reset_index(drop=True), resultado_df], axis=1))
    print(f"Endpoint router: {router_endpoint_name}")